# Patent Distribution Across MSA

In [1]:
import requests
import os
from bs4 import BeautifulSoup

import pandas as pd

import warnings
warnings.filterwarnings('ignore')

### Data Extraction

In [2]:
DATA_URL = "https://www.uspto.gov/web/offices/ac/ido/oeip/taf/cls_cbsa/allcbsa_gd.htm"
OUTPUT_DIR = "Data"

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [4]:
print("Fetching patent data from USPTO...")
response = requests.get(DATA_URL)
soup = BeautifulSoup(response.content, 'html.parser')

# Find the table
table = soup.find('table')
rows = table.find_all('tr')

# Parse table data
data = []
for row in rows[1:]:  # Skip header
    cols = row.find_all('td')
    if len(cols) >= 19:  # Ensure row has all columns
        try:
            regional_level = cols[0].text.strip()
            id_code = cols[1].text.strip()
            regional_title = cols[2].text.strip()
            year_2000 = float(cols[3].text.strip())
            year_2015 = float(cols[18].text.strip())
            
            data.append({
                'Regional_Level': regional_level,
                'CBSA_Code': id_code,
                'MSA_Name': regional_title,
                'Patents_2000': year_2000,
                'Patents_2015': year_2015
            })
        except (ValueError, AttributeError):
            continue
print("Patent data fetched successfully!!")

df = pd.DataFrame(data)
# Filter MSA data
df_metro = df[df['Regional_Level'] == 'Metropolitan Statistical Area'].copy()

Fetching patent data from USPTO...
Patent data fetched successfully!!


In [5]:
df_metro.info()

<class 'pandas.core.frame.DataFrame'>
Index: 375 entries, 0 to 374
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Regional_Level  375 non-null    object 
 1   CBSA_Code       375 non-null    object 
 2   MSA_Name        375 non-null    object 
 3   Patents_2000    375 non-null    float64
 4   Patents_2015    375 non-null    float64
dtypes: float64(2), object(3)
memory usage: 17.6+ KB


In [6]:
df_metro.describe()

,Patents_2000,Patents_2015
count,375.000000,375.000000
mean,427.930667,721.080000
std,4178.546009,7070.288837
min,0.000000,0.000000
25%,12.500000,13.000000
50%,34.000000,45.000000
75%,131.000000,165.500000
max,80230.000000,135192.000000


In [7]:
df_metro['Growth_Absolute'] = df_metro['Patents_2015'] - df_metro['Patents_2000']
df_metro['Growth_Percent'] = ((df_metro['Patents_2015'] - df_metro['Patents_2000']) / 
                               df_metro['Patents_2000'] * 100)

In [8]:
df_metro.info()

<class 'pandas.core.frame.DataFrame'>
Index: 375 entries, 0 to 374
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Regional_Level   375 non-null    object 
 1   CBSA_Code        375 non-null    object 
 2   MSA_Name         375 non-null    object 
 3   Patents_2000     375 non-null    float64
 4   Patents_2015     375 non-null    float64
 5   Growth_Absolute  375 non-null    float64
 6   Growth_Percent   373 non-null    float64
dtypes: float64(4), object(3)
memory usage: 23.4+ KB


In [10]:
df_metro[df_metro['Patents_2000'] == 0]

,Regional_Level,CBSA_Code,MSA_Name,Patents_2000,Patents_2015,Growth_Absolute,Growth_Percent
342,Metropolitan Statistical Area,110180,"Abilene, TX",0.0,8.0,8.0,inf
368,Metropolitan Statistical Area,125980,"Hinesville-Fort Stewart, GA",0.0,0.0,0.0,NaN
370,Metropolitan Statistical Area,141900,"San Germán-Cabo Rojo, PR",0.0,1.0,1.0,inf
371,Metropolitan Statistical Area,125020,"Guayama, PR",0.0,0.0,0.0,NaN


In [12]:
# Handle infinite/NaN values
df_metro['Growth_Percent'] = df_metro['Growth_Percent'].replace([float('inf'), -float('inf')], 0)
df_metro['Growth_Percent'] = df_metro['Growth_Percent'].fillna(0)

In [13]:
df_metro.info()

<class 'pandas.core.frame.DataFrame'>
Index: 375 entries, 0 to 374
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Regional_Level   375 non-null    object 
 1   CBSA_Code        375 non-null    object 
 2   MSA_Name         375 non-null    object 
 3   Patents_2000     375 non-null    float64
 4   Patents_2015     375 non-null    float64
 5   Growth_Absolute  375 non-null    float64
 6   Growth_Percent   375 non-null    float64
dtypes: float64(4), object(3)
memory usage: 23.4+ KB


In [14]:
print("Max : ", df_metro['CBSA_Code'].max())
print("Min : ", df_metro['CBSA_Code'].min())

Max :  149740
Min :  


In [15]:
df_metro.tail()

,Regional_Level,CBSA_Code,MSA_Name,Patents_2000,Patents_2015,Growth_Absolute,Growth_Percent
370,Metropolitan Statistical Area,141900,"San Germán-Cabo Rojo, PR",0.0,1.0,1.0,0.000000
371,Metropolitan Statistical Area,125020,"Guayama, PR",0.0,0.0,0.0,0.000000
372,Metropolitan Statistical Area,149500,"Yauco, PR",1.0,1.0,0.0,0.000000
373,Metropolitan Statistical Area,121940,"Fajardo, PR",1.0,0.0,-1.0,-100.000000
374,Metropolitan Statistical Area,,-- Subtotal --,80230.0,135192.0,54962.0,68.505547


In [16]:
df_metro = df_metro.iloc[:-1].copy()
df_metro.tail()

,Regional_Level,CBSA_Code,MSA_Name,Patents_2000,Patents_2015,Growth_Absolute,Growth_Percent
369,Metropolitan Statistical Area,138660,"Ponce, PR",1.0,0.0,-1.0,-100.0
370,Metropolitan Statistical Area,141900,"San Germán-Cabo Rojo, PR",0.0,1.0,1.0,0.0
371,Metropolitan Statistical Area,125020,"Guayama, PR",0.0,0.0,0.0,0.0
372,Metropolitan Statistical Area,149500,"Yauco, PR",1.0,1.0,0.0,0.0
373,Metropolitan Statistical Area,121940,"Fajardo, PR",1.0,0.0,-1.0,-100.0


In [17]:
print("Max : ", df_metro['CBSA_Code'].max())
print("Min : ", df_metro['CBSA_Code'].min())

Max :  149740
Min :  110180


In [20]:
df_metro['CBSA_Code'].map(type).value_counts()

CBSA_Code
<class 'str'>    374
Name: count, dtype: int64

In [21]:
df_metro['CBSA_Code'] = df_metro['CBSA_Code'].str[1:]
print("Max : ", df_metro['CBSA_Code'].max())
print("Min : ", df_metro['CBSA_Code'].min())
df_metro.head()

Max :  49740
Min :  10180


,Regional_Level,CBSA_Code,MSA_Name,Patents_2000,Patents_2015,Growth_Absolute,Growth_Percent
0,Metropolitan Statistical Area,41940,"San Jose-Sunnyvale-Santa Clara, CA",5812.0,14618.0,8806.0,151.514109
1,Metropolitan Statistical Area,35620,"New York-Northern New Jersey-Long Island, NY-N...",5689.0,7754.0,2065.0,36.298119
2,Metropolitan Statistical Area,41860,"San Francisco-Oakland-Fremont, CA",3625.0,9732.0,6107.0,168.468966
3,Metropolitan Statistical Area,31100,"Los Angeles-Long Beach-Santa Ana, CA",3877.0,6476.0,2599.0,67.036368
4,Metropolitan Statistical Area,14460,"Boston-Cambridge-Quincy, MA-NH",2972.0,5949.0,2977.0,100.168237


In [22]:
df_final = df_metro[[
    'CBSA_Code',
    'MSA_Name',
    'Patents_2000',
    'Patents_2015',
    'Growth_Absolute',
    'Growth_Percent'
]]

output_file = 'Data/final_patent_data.csv'
df_final.to_csv(output_file, index=False)

print(f"\n{'='*80}")
print(f"Data prepared successfully!")
print(f"{'='*80}")
print(f"\nOutput file: {output_file}")
print(f"Total Metropolitan Statistical Areas: {len(df_final)}")
print(f"\nData columns:")
for col in df_final.columns:
    print(f"  - {col}")


Data prepared successfully!

Output file: Data/final_patent_data.csv
Total Metropolitan Statistical Areas: 374

Data columns:
  - CBSA_Code
  - MSA_Name
  - Patents_2000
  - Patents_2015
  - Growth_Absolute
  - Growth_Percent
